# TAVE 18기 DRAG 스터디 - 3주차 RAG 챗봇 + SigLIP 이미지 검색

2주차 챗봇(Dense 검색 + LLM)에 **Reranking**을 붙이고, 켰을 때와 껐을 때를 비교한다.
여기에 4주차 **SigLIP**을 붙여서, 질문하면 문서 답변과 함께 **관련 이미지도 찾아서** 보여준다 (5주차 텍스트→이미지 검색 미리보기).

| 모드 | 동작 |
|---|---|
| 기본 검색 | bge-m3 Dense 검색 상위 4개 → LLM 답변 (2주차와 동일) |
| Reranking | Dense 상위 20개 → Cross-Encoder(bge-reranker-v2-m3)로 다시 채점 → 상위 4개 → LLM 답변 |
| 이미지 검색 (SigLIP) | 질문을 SigLIP 텍스트 인코더로 벡터화 → 미리 저장한 이미지 벡터와 코사인 유사도 → 상위 이미지 + 짝일 확률 |

**이전 버전에서 바뀐 점**
- 기법 8개 → **기본 검색 / Reranking 2개**만 남김 (HyDE·Multi Query·Self-RAG 등은 제거, BM25·Kiwi 의존성도 제거)
- 웹 UI를 **대화형 채팅**으로 변경: 답변이 실시간으로 나오고, 이전 대화를 기억하고, 답변마다 근거 문서를 접어서 보여줌
- 문서 추가·선택을 채팅 화면 옆 사이드바에서 바로 처리
- 평가는 UI에서 빼고 노트북 셀로 정리: **기본 vs Reranking** 두 모드만 RAGAS 4대 지표로 비교
- **SigLIP 이미지 검색 추가**: 사이드바에서 이미지를 올리면 질문할 때 관련 이미지를 같이 보여주고, `🖼️ 이미지 검색` 탭에서 문장으로 이미지만 찾을 수도 있음.
  한국어 질문을 쓰기 위해 다국어 체크포인트(`siglip-base-patch16-256-multilingual`)를 사용

**실행 방법**: `런타임 → 런타임 유형 변경 → GPU(T4 이상)` 선택 후 위에서부터 차례대로 실행.
평가(5번)는 필요할 때만 돌리면 되고, 챗봇만 쓰려면 5번을 건너뛰고 6번을 실행하면 된다.

In [ ]:
# 설치 (Colab에는 torch / transformers / pandas / matplotlib 가 이미 있음)
!pip -q install -U sentence-transformers langchain-text-splitters faiss-cpu pymupdf gradio beautifulsoup4 accelerate sentencepiece

In [ ]:
# ------------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------------
import os, re, json, time
from threading import Thread
import numpy as np
import pandas as pd
import torch

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # 느리거나 메모리가 부족하면 "Qwen/Qwen2.5-1.5B-Instruct"
EMB_ID = "BAAI/bge-m3"                     # 2주차와 동일한 임베딩
RERANK_ID = "BAAI/bge-reranker-v2-m3"      # 다국어 Cross-Encoder
SIGLIP_ID = "google/siglip-base-patch16-256-multilingual"  # 4주차 SigLIP (한국어 질문용 다국어 모델)

CHUNK_SIZE, CHUNK_OVERLAP = 800, 100
K_FINAL = 4          # LLM에 넘길 문맥 수
K_FIRST = 20         # Reranking 전에 넓게 가져올 후보 수
ANSWER_TOKENS = 400
HISTORY_TURNS = 2    # 답변할 때 참고할 이전 대화 수 (질문+답변 = 1턴)
GEN_BATCH = 4        # 평가할 때 한 번에 생성할 프롬프트 수
IMG_TOP_K = 3        # 질문마다 보여줄 이미지 수
IMG_MIN_PROB = 0.05  # SigLIP '짝일 확률'이 이 값보다 낮은 이미지는 숨김 (4주차: 정답도 20~30%대로 낮게 나옴)

WORKDIR = "/content/drag_week3" if os.path.isdir("/content") else "./drag_week3"
os.makedirs(WORKDIR, exist_ok=True)
TESTSET_PATH = f"{WORKDIR}/testset.csv"
RESULTS_PATH = f"{WORKDIR}/eval_results.jsonl"
IMG_DIR = f"{WORKDIR}/images"
os.makedirs(IMG_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# T4는 bf16 미지원 → fp16, Ampere(L4/A100) 이상은 bf16
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", DTYPE)

MODES = {"baseline": "기본 검색", "rerank": "Reranking"}

## 1. 모델 로드 (LLM / 임베딩 / Reranker)
- `stream()`: 채팅용. 답변을 한 글자씩 흘려보낸다
- `generate()`: 평가용. 여러 프롬프트를 묶어서 한 번에 생성한다

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from sentence_transformers import SentenceTransformer, CrossEncoder

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"  # 배치 생성 시 왼쪽 패딩
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map=DEVICE).eval()

embedder = SentenceTransformer(EMB_ID, device=DEVICE)
reranker = CrossEncoder(RERANK_ID, max_length=512, device=DEVICE)
if DEVICE == "cuda":
    embedder.half()
    reranker.model.half()

GEN_KW = dict(do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)


def _to_text(messages):
    """str이면 user 메시지 1개로, list면 그대로 채팅 템플릿에 넣는다."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _clean(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


@torch.inference_mode()
def generate(prompts, max_new_tokens=256):
    """프롬프트(str 또는 메시지 리스트) 1개 또는 여러 개를 greedy 생성."""
    single = isinstance(prompts, str) or (prompts and isinstance(prompts[0], dict))
    ps = [prompts] if single else list(prompts)
    outs = []
    for i in range(0, len(ps), GEN_BATCH):
        enc = tok([_to_text(p) for p in ps[i:i + GEN_BATCH]], return_tensors="pt", padding=True).to(llm.device)
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, **GEN_KW)
        outs += [_clean(tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)) for o in out]
    return outs[0] if single else outs


def stream(messages, max_new_tokens=ANSWER_TOKENS):
    """답변을 생성하면서 지금까지의 텍스트를 계속 내보낸다 (채팅 화면용)."""
    enc = tok(_to_text(messages), return_tensors="pt").to(llm.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    Thread(target=llm.generate, kwargs=dict(**enc, streamer=streamer, max_new_tokens=max_new_tokens, **GEN_KW)).start()
    text = ""
    for piece in streamer:
        text += piece
        yield _clean(text)


def embed(texts):
    return embedder.encode(texts, batch_size=32, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False).astype("float32")


print(generate("RAG를 한 문장으로 설명해줘.", 80))
if DEVICE == "cuda":
    print(f"GPU 메모리 사용: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 1-2. SigLIP 로드 (이미지 검색용)
4주차 실습과 같은 코드. 이미지와 문장을 같은 공간의 벡터로 바꾼다.
- `embed_images()`: 이미지 → 벡터 (미리 한 번만 계산해서 저장)
- `embed_siglip_text()`: 질문 → 벡터 (질문마다 계산). 학습 때처럼 `padding="max_length"` 필수
- 확률 = `sigmoid(t · 코사인 + b)`. t, b는 모델이 학습한 값을 그대로 쓴다

In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image

sig_processor = AutoProcessor.from_pretrained(SIGLIP_ID)
sig_model = AutoModel.from_pretrained(SIGLIP_ID, dtype=DTYPE).to(DEVICE).eval()
SIG_T = sig_model.logit_scale.exp().item()  # 온도 t
SIG_B = sig_model.logit_bias.item()         # 바이어스 b


def _vec(out):
    """transformers 5.x는 출력 객체(.pooler_output), 4.x는 텐서를 돌려준다. 둘 다 처리하고 L2 정규화."""
    out = getattr(out, "pooler_output", out)
    return torch.nn.functional.normalize(out.float(), dim=-1).cpu().numpy()


@torch.inference_mode()
def embed_images(images, batch=16):
    vecs = []
    for i in range(0, len(images), batch):
        x = sig_processor(images=images[i:i + batch], return_tensors="pt").to(DEVICE)
        vecs.append(_vec(sig_model.get_image_features(pixel_values=x["pixel_values"].to(sig_model.dtype))))
    return np.vstack(vecs).astype("float32")


@torch.inference_mode()
def embed_siglip_text(texts):
    x = sig_processor(text=texts, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    return _vec(sig_model.get_text_features(**x)).astype("float32")


print(f"SigLIP 로드 완료 · t = {SIG_T:.2f}, b = {SIG_B:.2f}")

## 2. 문서 로더 + 지식베이스
- URL: 웹페이지는 본문만 추출(위키는 `#mw-content-text`), PDF 링크는 PyMuPDF로 페이지별 텍스트 추출
- 문서마다 따로 읽고 청크에 `출처 / 제목 / 페이지`를 붙인다
- 검색: bge-m3 임베딩 + FAISS 내적(= 코사인 유사도)

In [ ]:
import requests, faiss
import pymupdf
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

UA = {"User-Agent": "Mozilla/5.0 (TAVE-DRAG-study)"}


def load_pdf_bytes(data, source, title=None):
    doc = pymupdf.open(stream=data, filetype="pdf")
    first_line = next((l.strip() for l in (doc[0].get_text().splitlines() if len(doc) else []) if len(l.strip()) > 10), "")
    title = title or (doc.metadata or {}).get("title") or first_line[:80] or os.path.basename(source)
    pages = [{"text": p.get_text().strip(), "page": i + 1} for i, p in enumerate(doc)]
    pages = [p for p in pages if len(p["text"]) >= 30]  # 텍스트 레이어가 없는 스캔 페이지는 건너뜀
    return {"source": source, "title": title, "type": "pdf", "pages": pages}


def load_url(url):
    r = requests.get(url, headers=UA, timeout=60)
    r.raise_for_status()
    if "charset" not in r.headers.get("content-type", ""):
        r.encoding = r.apparent_encoding  # 인코딩을 안 알려주는 사이트에서 한글이 깨지지 않게
    if "pdf" in r.headers.get("content-type", "") or r.content[:4] == b"%PDF":
        return load_pdf_bytes(r.content, source=url)
    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.title.get_text().strip() if soup.title else url
    title = re.sub(r"\s*[-–|]\s*(위키백과|Wikipedia).*$", "", title)
    for t in soup(["script", "style", "nav", "footer", "header", "aside", "noscript", "sup", "table", "figure"]):
        t.decompose()
    for t in soup.select(".mw-editsection, #toc, .toc, .navbox, .sidebar, .infobox, .hatnote, .ambox, "
                         ".metadata, .reflist, .mw-references-wrap, .references, .catlinks"):
        t.decompose()
    main = soup.select_one("#mw-content-text") or soup.find("article") or soup.find("main") or soup.body or soup
    # 링크마다 줄이 끊기지 않도록 문단/제목/목록 단위로 텍스트를 모은다
    blocks = [b.get_text(" ", strip=True) for b in main.find_all(["h1", "h2", "h3", "h4", "p", "li", "dd", "blockquote", "pre"])
              if not b.find_parent(["li", "blockquote"])]
    text = "\n\n".join(re.sub(r"\s+", " ", b) for b in blocks if b) or main.get_text("\n")
    text = re.sub(r" ([.,)])", r"\1", text).strip()
    return {"source": url, "title": title, "type": "web", "pages": [{"text": text, "page": None}]}


splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)


class KnowledgeBase:
    def __init__(self):
        self.docs = {}    # source -> 문서 정보 + 청크 + 임베딩
        self.chunks = []  # 전체 청크 (id = 리스트 인덱스)
        self.index = None

    def add(self, doc):
        if not doc["pages"]:
            raise ValueError("추출된 텍스트가 없습니다 (스캔 PDF는 지원하지 않음)")
        doc["pieces"] = [(p["page"], t) for p in doc["pages"] for t in splitter.split_text(p["text"])]
        doc["emb"] = embed([t for _, t in doc["pieces"]])
        self.docs[doc["source"]] = doc
        self._rebuild()
        return len(doc["pieces"])

    def remove(self, source):
        self.docs.pop(source, None)
        self._rebuild()

    def _rebuild(self):
        self.chunks, embs = [], []
        for di, (src, d) in enumerate(self.docs.items(), start=1):
            d["label"] = f"D{di}"
            for page, text in d["pieces"]:
                self.chunks.append({"id": len(self.chunks), "text": text, "source": src,
                                    "title": d["title"], "label": d["label"], "page": page})
            embs.append(d["emb"])
        self.index = None
        if self.chunks:
            mat = np.vstack(embs)
            self.index = faiss.IndexFlatIP(mat.shape[1])  # 정규화된 벡터의 내적 = 코사인 유사도
            self.index.add(mat)

    def search(self, query, k, sources=None):
        """[(청크 id, 코사인 유사도)] 상위 k개. sources를 주면 그 문서들 안에서만 찾는다."""
        scores, ids = self.index.search(embed([query]), len(self.chunks))
        hits = [(int(i), float(s)) for i, s in zip(ids[0], scores[0])
                if not sources or self.chunks[i]["source"] in sources]
        return hits[:k]


KB = KnowledgeBase()

### 2-2. 이미지 저장소 (SigLIP)
올린 이미지를 `IMG_DIR`에 복사하고 벡터를 미리 계산해 둔다. 질문이 오면 질문 벡터만 만들어서 비교한다 (4주차 '사진은 미리, 질문만 그때그때').

In [ ]:
import shutil


class ImageBase:
    def __init__(self):
        self.items = []  # [{"path", "name", "emb"}]

    def add(self, paths):
        """이미지 파일 경로들을 받아 저장. 추가한 개수를 돌려준다."""
        new = []
        for p in paths:
            try:
                img = Image.open(p).convert("RGB")
            except Exception as e:
                print(f"❌ {os.path.basename(p)}: {e}")
                continue
            dst = os.path.join(IMG_DIR, os.path.basename(p))
            if os.path.abspath(p) != os.path.abspath(dst):
                shutil.copy(p, dst)
            new.append((dst, img))
        if new:
            embs = embed_images([img for _, img in new])
            names = {it["name"] for it in self.items}
            for (dst, _), e in zip(new, embs):
                if os.path.basename(dst) not in names:
                    self.items.append({"path": dst, "name": os.path.basename(dst), "emb": e})
        return len(new)

    def clear(self):
        self.items = []

    def search(self, query, k=IMG_TOP_K, min_prob=0.0):
        """[{path, name, cos, prob}] 코사인 유사도 상위 k개 중 확률이 min_prob 이상인 것만."""
        if not self.items:
            return []
        cos = np.vstack([it["emb"] for it in self.items]) @ embed_siglip_text([query])[0]
        prob = 1 / (1 + np.exp(-(SIG_T * cos + SIG_B)))
        return [{**{k_: self.items[i][k_] for k_ in ("path", "name")}, "cos": float(cos[i]), "prob": float(prob[i])}
                for i in np.argsort(-cos)[:k] if prob[i] >= min_prob]


IMB = ImageBase()

## 3. 검색 + 답변
`retrieve()`가 두 모드의 유일한 차이다. 답변 프롬프트에는 이전 대화 몇 턴과 번호 붙은 문맥을 넣고, 답변 문장 끝에 `[1]`처럼 근거 번호를 달게 한다.

In [ ]:
def rerank(query, cids):
    """Cross-Encoder로 (질문, 문서) 쌍을 직접 채점. 점수는 0~1."""
    s = np.array(reranker.predict([(query, KB.chunks[c]["text"]) for c in cids],
                                  batch_size=16, show_progress_bar=False), dtype=float)
    if s.min() < 0 or s.max() > 1:  # logit이면 0~1로 변환
        s = 1 / (1 + np.exp(-s))
    return s


def retrieve(query, mode="rerank", sources=None):
    """[{cid, score, dense_rank}] 상위 K_FINAL개. score는 기본 검색이면 코사인, Reranking이면 rerank 점수."""
    if mode == "baseline":
        return [{"cid": c, "score": s, "dense_rank": i} for i, (c, s) in enumerate(KB.search(query, K_FINAL, sources), 1)]
    hits = KB.search(query, K_FIRST, sources)
    scores = rerank(query, [c for c, _ in hits])
    order = np.argsort(-scores)[:K_FINAL]
    return [{"cid": hits[i][0], "score": float(scores[i]), "dense_rank": int(i) + 1} for i in order]


def cite(cid):
    c = KB.chunks[cid]
    return f"{c['label']} {c['title'][:25]}" + (f" p.{c['page']}" if c["page"] else "")


def short(t, n=120):
    t = re.sub(r"\s+", " ", t)
    return t[:n] + "…" if len(t) > n else t


SYSTEM_PROMPT = """당신은 사용자가 올린 문서를 바탕으로 질문에 답하는 친절한 한국어 AI 어시스턴트입니다.
- 질문과 함께 주어지는 [문맥]의 내용만 사용해서 답하세요.
- 문장 끝에 근거로 쓴 문맥 번호를 [1], [2]처럼 붙이세요.
- 문맥에서 답을 찾을 수 없으면 "주어진 문서에서 답을 찾을 수 없습니다."라고만 답하세요.
- 기술 용어나 이름은 번역하지 말고 그대로 쓰고, 3~5문장으로 간결하게 답하세요."""


def format_context(hits):
    return "\n\n".join(f"[{i}] (출처: {cite(h['cid'])})\n{KB.chunks[h['cid']]['text']}" for i, h in enumerate(hits, 1)) \
        or "(검색된 문서 없음)"


def _text(content):
    """Gradio 채팅 기록은 content가 [{"type": "text", "text": ...}] 형태로 올 수 있어서 문자열로 바꾼다."""
    if isinstance(content, str):
        return content
    return "\n".join(c.get("text", "") for c in content if isinstance(c, dict) and c.get("type") == "text")


def build_messages(question, hits, history=()):
    """history: [{"role", "content"}] 이전 대화. 근거 표시용 메시지(metadata 있음)는 빼고 최근 HISTORY_TURNS턴만 쓴다."""
    turns = [{"role": m["role"], "content": _text(m["content"])} for m in history
             if m["role"] in ("user", "assistant") and not m.get("metadata")]
    # 이미지 결과(🖼️로 시작)와 글이 없는 메시지는 LLM에 넘기지 않는다
    turns = [t for t in turns if t["content"].strip() and not t["content"].startswith("🖼️")]
    return ([{"role": "system", "content": SYSTEM_PROMPT}] + turns[-2 * HISTORY_TURNS:]
            + [{"role": "user", "content": f"[문맥]\n{format_context(hits)}\n\n[질문]\n{question}"}])


def answer(question, mode="rerank", sources=None):
    """노트북에서 바로 쓰는 한 번짜리 질문 함수."""
    hits = retrieve(question, mode, sources)
    return generate(build_messages(question, hits), ANSWER_TOKENS), hits

## 4. 예시 문서·이미지 불러오기 + 동작 확인
- 문서: 성격이 다른 링크 3개. 한국어 위키 2개(주제가 비슷해서 검색이 헷갈리기 쉬움) + 영어 논문 PDF(한국어 질문 ↔ 영어 문서)
- 이미지: 4주차 발표에 쓴 사진 4장 + 4주차 실습의 COCO 고양이 사진. 한국어 질문으로도 찾아지는지 확인

In [ ]:
SAMPLE_URLS = [
    "https://ko.wikipedia.org/wiki/대형_언어_모델",
    "https://ko.wikipedia.org/wiki/자연어_처리",
    "https://arxiv.org/pdf/2005.11401",  # Lewis et al., 2020 - RAG 원 논문
]
for u in SAMPLE_URLS:
    try:
        print(f"✅ {u} → 청크 {KB.add(load_url(u))}개")
    except Exception as e:
        print(f"❌ {u} → {e}")
print("전체 청크 수:", len(KB.chunks))

# 예시 이미지: 4주차 발표에 쓴 사진들 (scikit-image 샘플, CC0/퍼블릭 도메인) + COCO 고양이 두 마리
from skimage import data as skdata
for name, fn in {"cat": skdata.chelsea, "coffee": skdata.coffee, "rocket": skdata.rocket, "astronaut": skdata.astronaut}.items():
    Image.fromarray(fn()).save(f"{IMG_DIR}/{name}.jpg")
try:
    r = requests.get("http://images.cocodataset.org/val2017/000000039769.jpg", headers=UA, timeout=60)
    open(f"{IMG_DIR}/coco_two_cats.jpg", "wb").write(r.content)
except Exception as e:
    print("COCO 이미지 다운로드 실패:", e)
print("이미지", IMB.add(sorted(os.path.join(IMG_DIR, f) for f in os.listdir(IMG_DIR))), "장 저장")
for q_img in ["소파 위에서 자는 고양이 두 마리", "밤에 발사되는 로켓", "a cup of coffee"]:
    print(f"\n🔎 {q_img}")
    for h in IMB.search(q_img):
        print(f"  {h['name']:<20} 코사인 {h['cos']:.3f} · 짝일 확률 {h['prob']:.1%}")

q = "RAG 모델에서 retriever와 generator는 각각 무엇을 사용했나?"
for mode, name in MODES.items():
    ans, hits = answer(q, mode)
    print(f"\n===== {name}\n{ans}")
    for i, h in enumerate(hits, 1):
        print(f"  [{i}] {cite(h['cid'])}  점수 {h['score']:.3f}  (Dense {h['dense_rank']}위)")

## 5. (선택) 평가: 기본 검색 vs Reranking
문서마다 청크를 골고루 뽑아 Qwen이 질문·정답을 만들고, 두 모드의 답변을 RAGAS 4대 지표 정의대로 채점한다.

| 지표 | 계산 방식 (RAGAS 정의) |
|---|---|
| Faithfulness | 답변을 사실 진술로 쪼갬 → 각 진술이 문맥에서 추론 가능한지 → 지지된 비율 |
| Answer Relevancy | 답변만 보고 질문 3개를 거꾸로 생성 → 원래 질문과의 임베딩 코사인 평균 (회피성 답변이면 0) |
| Context Precision | 각 문맥이 정답 도출에 유용했는지 → 순위를 반영한 Average Precision |
| Context Recall | 정답을 문장으로 쪼갬 → 각 문장이 문맥에서 뒷받침되는지 → 비율 |

⚠️ 채점자도 소형 LLM이라 절대값보다 **두 모드의 상대 비교**로 보는 게 안전하다. 결과는 문항마다 `eval_results.jsonl`에 저장돼서 런타임이 끊겨도 이어서 돌릴 수 있다.

In [ ]:
N_Q_PER_DOC = 3  # 문서당 질문 수

TESTSET_PROMPT = """다음 문서 조각만 보고 답할 수 있는 질문 1개와 정답을 만드세요.
- 질문과 정답은 반드시 한국어로 작성 (기술 용어·이름은 원문 그대로 사용 가능)
- 문서 조각에 있는 구체적 사실(정의, 인물, 연도, 수치, 방법 등)을 묻는 질문
- "이 문서에서", "위 내용에서" 같은 표현 없이, 질문만 보고도 무엇을 묻는지 알 수 있게
- 정답은 1~3문장, 문서 조각의 내용만으로 작성
JSON으로만 출력: {{"question": "...", "ground_truth": "..."}}

#문서 조각:
{chunk}"""


def parse_json(text, want=dict):
    text = re.sub(r"```(json)?", "", text)
    for pat in ([r"\{.*\}", r"\[.*\]"] if want is dict else [r"\[.*\]", r"\{.*\}"]):
        m = re.search(pat, text, re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return None


def make_testset(n_per_doc=N_Q_PER_DOC):
    picks = []
    for src in KB.docs:
        ids = [c["id"] for c in KB.chunks if c["source"] == src and len(c["text"]) >= 400
               and c["text"].count("et al") < 2]  # 참고문헌 청크는 제외
        if ids:
            picks += [ids[int(i)] for i in np.linspace(0, len(ids) - 1, n_per_doc + 2)[1:-1]]
    rows = []
    for cid, o in zip(picks, generate([TESTSET_PROMPT.format(chunk=KB.chunks[c]["text"]) for c in picks], 300)):
        j = parse_json(o) or {}
        if j.get("question") and j.get("ground_truth"):
            ch = KB.chunks[cid]
            rows.append({"qid": f"{ch['label']}-{len(rows) + 1}", "doc": ch["label"], "question": j["question"],
                         "ground_truth": j["ground_truth"], "ref_chunk": ch["text"]})
    return pd.DataFrame(rows)


testset = make_testset()
testset.to_csv(TESTSET_PATH, index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", 200)
testset[["qid", "question", "ground_truth"]]

### ✋ 검수
생성된 질문·정답이 이상하면 아래 셀에서 고치거나, 왼쪽 파일 탭에서 `drag_week3/testset.csv`를 내려받아 고친 뒤 다시 올리고 `read_csv` 줄을 실행한다.

In [ ]:
# testset.loc[testset.qid == "D1-1", "ground_truth"] = "수정한 정답"
# testset = testset[testset.qid != "D2-2"]            # 질문 삭제
# testset = pd.read_csv(TESTSET_PATH)                 # CSV로 수정했다면 다시 읽기
testset.to_csv(TESTSET_PATH, index=False, encoding="utf-8-sig")
print(len(testset), "문항 저장")

In [ ]:
FAITH_PROMPT = """답변을 독립적인 사실 진술(statement)들로 나누고, 각 진술이 문맥으로부터 직접 추론 가능한지 판정하세요.
추론 가능하면 verdict=1, 아니면 0.
JSON 배열로만 출력: [{{"statement": "...", "verdict": 1}}]

#문맥:
{context}

#답변:
{answer}"""

AR_PROMPT = """아래 답변을 보고, 이 답변이 대답하고 있을 법한 질문을 한국어로 3개 만드세요.
답변이 "모르겠다", "찾을 수 없다"처럼 회피성이면 noncommittal=1, 아니면 0.
JSON으로만 출력: {{"questions": ["...", "...", "..."], "noncommittal": 0}}

#답변:
{answer}"""

CP_PROMPT = """아래 번호가 매겨진 문맥들 각각이 질문의 정답을 도출하는 데 유용했는지 판정하세요.
유용하면 1, 아니면 0. 문맥 개수({n}개)와 같은 길이의 JSON 배열로만 출력: 예) [1, 0, 1]

#질문:
{question}

#정답:
{ground_truth}

#문맥:
{context}"""

CR_PROMPT = """정답을 문장 단위로 나누고, 각 문장의 내용이 문맥에서 뒷받침되는지 판정하세요.
뒷받침되면 attributed=1, 아니면 0.
JSON 배열로만 출력: [{{"sentence": "...", "attributed": 1}}]

#문맥:
{context}

#정답:
{ground_truth}"""

METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]


def _binary_list(text, key, n=None):
    j = parse_json(text, want=list)
    vals = []
    for x in j if isinstance(j, list) else []:
        try:
            vals.append(1 if int(x.get(key) if isinstance(x, dict) else x) == 1 else 0)
        except Exception:
            pass
    if not vals:  # JSON이 깨졌을 때 정규식으로 복구
        vals = [int(v) for v in re.findall(rf'"{key}"\s*:\s*([01])' if key else r"\b[01]\b", text)]
    return vals[:n] if n else vals


def average_precision(v):
    hit, s = 0, 0.0
    for k, x in enumerate(v, 1):
        if x:
            hit += 1
            s += hit / k
    return s / sum(v) if sum(v) else 0.0


def judge(samples):
    """samples: [{question, ground_truth, answer, contexts(list[str])}] → 각 sample에 4개 지표 점수 추가"""
    jobs = []
    for i, s in enumerate(samples):
        ctx = "\n\n".join(f"[{k}] {t}" for k, t in enumerate(s["contexts"], 1))
        s["refused"] = "찾을 수 없" in s["answer"]
        s["faithfulness"] = float("nan")  # 답을 거부했으면 충실도는 계산하지 않음
        jobs += [(i, "answer_relevancy", AR_PROMPT.format(answer=s["answer"]), 150),
                 (i, "context_precision", CP_PROMPT.format(question=s["question"], ground_truth=s["ground_truth"],
                                                           context=ctx, n=len(s["contexts"])), 60),
                 (i, "context_recall", CR_PROMPT.format(context=ctx, ground_truth=s["ground_truth"]), 400)]
        if not s["refused"]:
            jobs.append((i, "faithfulness", FAITH_PROMPT.format(context=ctx, answer=s["answer"]), 500))
    outs = generate([j[2] for j in jobs], max(j[3] for j in jobs))
    for (i, metric, _, _), o in zip(jobs, outs):
        s = samples[i]
        if metric in ("faithfulness", "context_recall"):
            v = _binary_list(o, "verdict" if metric == "faithfulness" else "attributed")
            s[metric] = float(np.mean(v)) if v else float("nan")
        elif metric == "context_precision":
            v = _binary_list(o, None, len(s["contexts"]))
            s[metric] = average_precision(v) if v else float("nan")
        else:  # answer_relevancy
            j = parse_json(o, want=dict) or {}
            qs = [q for q in j.get("questions", []) if isinstance(q, str) and q.strip()]
            if s["refused"] or j.get("noncommittal") in (1, "1") or not qs:
                s[metric] = 0.0
            else:
                e = embed([s["question"]] + qs)
                s[metric] = float(np.mean(e[1:] @ e[0]))
    return samples


def run_eval(testset, path=RESULTS_PATH):
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            done = {(r["qid"], r["mode"]) for r in map(json.loads, f)}
    for n, row in enumerate(testset.itertuples(index=False), 1):
        todo = [m for m in MODES if (row.qid, m) not in done]
        if not todo:
            continue
        t0 = time.time()
        hits = {m: retrieve(row.question, m) for m in todo}
        answers = generate([build_messages(row.question, hits[m]) for m in todo], ANSWER_TOKENS)
        samples = [{"qid": row.qid, "doc": row.doc, "mode": m, "question": row.question,
                    "ground_truth": row.ground_truth, "answer": a,
                    "contexts": [KB.chunks[h["cid"]]["text"] for h in hits[m]],
                    "context_docs": [cite(h["cid"]) for h in hits[m]]} for m, a in zip(todo, answers)]
        judge(samples)
        with open(path, "a", encoding="utf-8") as f:
            for s in samples:
                f.write(json.dumps(s, ensure_ascii=False) + "\n")
        print(f"[{n}/{len(testset)}] {row.qid} 완료 ({time.time() - t0:.0f}s)")
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])


def summarize(df):
    t = df.groupby("mode")[METRICS].mean()
    t["mean"] = t[METRICS].mean(axis=1)
    t["refusal_rate"] = df.groupby("mode")["refused"].mean()
    return t.reindex([m for m in MODES if m in t.index]).rename(index=MODES).round(3)


def plot_summary(table):
    import matplotlib.pyplot as plt
    cols = METRICS + ["mean"]
    x = np.arange(len(cols))
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for i, (name, row) in enumerate(table[cols].iterrows()):
        label = {"기본 검색": "Baseline"}.get(name, name)  # Colab matplotlib에는 한글 폰트가 없어서 영어로 표시
        bars = ax.bar(x + (i - 0.5) * 0.38, row.values, 0.36, label=label, color=["#94A3B8", "#4F46E5"][i % 2])
        ax.bar_label(bars, fmt="%.2f", fontsize=8)
    ax.set_xticks(x, cols)
    ax.set_ylim(0, 1.1)
    ax.legend(loc="upper right")
    ax.set_title("Baseline vs Reranking (RAGAS-style)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    return fig

In [ ]:
results = run_eval(testset)
summary = summarize(results)
display(summary)
plot_summary(summary);

## 6. 웹 챗봇 (Gradio)
실행하면 `https://xxxx.gradio.live` 주소가 나온다. 런타임이 살아 있는 동안만 접속할 수 있다.
- **💬 채팅**: 답변이 실시간으로 나오고, 이전 대화를 기억한다. 답변 위의 `📎 근거 문서`를 펼치면 어떤 문서의 몇 페이지를 썼는지 보인다
- **사이드바**: 문서 추가(URL / PDF), 검색할 문서 고르기, 이미지 추가, Reranking·이미지 검색 켜고 끄기
- **🖼️ 이미지 검색**: 문장으로 이미지만 찾는다 (SigLIP 짝일 확률 표시). 5주차 과제의 미리보기
- **⚖️ 비교**: 같은 질문을 기본 검색 / Reranking으로 동시에 돌려서 나란히 본다 (Reranking으로 순위가 어떻게 바뀌었는지 포함)

In [ ]:
import gradio as gr


def doc_choices():
    return [(f"{d['label']} · {d['title'][:28]} ({len(d['pieces'])}청크)", src) for src, d in KB.docs.items()]


def sources_md(hits, mode):
    lines = []
    for i, h in enumerate(hits, 1):
        moved = f" · Dense {h['dense_rank']}위 → {i}위" if mode == "rerank" else ""
        lines.append(f"**[{i}] {cite(h['cid'])}** · 점수 {h['score']:.2f}{moved}\n\n> {short(KB.chunks[h['cid']]['text'], 160)}")
    return "\n\n".join(lines)


def ui_add_docs(urls, files):
    msgs = []
    for u in [u.strip() for u in (urls or "").splitlines() if u.strip()]:
        try:
            msgs.append(f"✅ {short(u, 40)} ({KB.add(load_url(u))}청크)")
        except Exception as e:
            msgs.append(f"❌ {short(u, 40)}: {e}")
    for f in files or []:
        path = f if isinstance(f, str) else getattr(f, "name", str(f))
        try:
            with open(path, "rb") as fh:
                msgs.append(f"✅ {os.path.basename(path)} ({KB.add(load_pdf_bytes(fh.read(), os.path.basename(path)))}청크)")
        except Exception as e:
            msgs.append(f"❌ {os.path.basename(path)}: {e}")
    ch = doc_choices()
    return ("\n\n".join(msgs) or "추가할 URL이나 PDF를 넣어 주세요.", gr.CheckboxGroup(choices=ch, value=[s for _, s in ch]),
            "", None)


def ui_remove_docs(selected):
    for src in selected or []:
        KB.remove(src)
    ch = doc_choices()
    return f"🗑️ {len(selected or [])}개 삭제", gr.CheckboxGroup(choices=ch, value=[s for _, s in ch])


def image_caption(h):
    return f"🖼️ {h['name']} · 짝일 확률 {h['prob']:.1%} (코사인 {h['cos']:.2f})"


def ui_chat(message, history, use_rerank, selected, use_images, min_prob):
    history = list(history or [])
    if not message.strip():
        yield history, ""
        return
    history.append({"role": "user", "content": message})
    use_docs = bool(KB.chunks and selected)
    imgs = IMB.search(message, IMG_TOP_K, min_prob) if use_images else []
    if not use_docs and not (use_images and IMB.items):
        history.append({"role": "assistant", "content": "먼저 왼쪽 사이드바에서 문서나 이미지를 추가하고, 검색할 문서를 체크해 주세요."})
        yield history, ""
        return
    if use_docs:
        mode = "rerank" if use_rerank else "baseline"
        hits = retrieve(message, mode, selected)
        messages = build_messages(message, hits, history[:-1])
        history.append({"role": "assistant", "content": sources_md(hits, mode),
                        "metadata": {"title": f"📎 근거 문서 {len(hits)}개 · {MODES[mode]}", "status": "done"}})
        history.append({"role": "assistant", "content": "…"})
        yield history, ""
        for text in stream(messages):
            history[-1]["content"] = text
            yield history, ""
    if use_images and IMB.items:
        if imgs:
            history.append({"role": "assistant", "content": f"🖼️ 관련 이미지 {len(imgs)}장 (SigLIP)"})
            for h in imgs:
                history.append({"role": "assistant", "content": {"path": h["path"], "alt_text": image_caption(h)}})
                history.append({"role": "assistant", "content": image_caption(h)})
        else:
            history.append({"role": "assistant", "content": f"🖼️ 짝일 확률 {min_prob:.0%} 이상인 이미지가 없습니다."})
        yield history, ""


def ui_add_images(files):
    paths = [f if isinstance(f, str) else getattr(f, "name", str(f)) for f in files or []]
    n = IMB.add(paths)
    return f"🖼️ {n}장 추가 · 전체 {len(IMB.items)}장", [(it["path"], it["name"]) for it in IMB.items], None


def ui_clear_images():
    IMB.clear()
    return "🗑️ 이미지를 모두 지웠습니다.", []


def ui_image_search(query, min_prob):
    if not IMB.items:
        return [], "먼저 사이드바에서 이미지를 추가해 주세요."
    if not query.strip():
        return [], "찾을 내용을 문장으로 입력해 주세요."
    hits = IMB.search(query, len(IMB.items), 0.0)
    shown = [h for h in hits if h["prob"] >= min_prob]
    info = (f"전체 {len(hits)}장 중 짝일 확률 {min_prob:.0%} 이상 {len(shown)}장. "
            "SigLIP 확률은 쌍마다 따로 계산돼서 정답도 낮게 나올 수 있으니, 순위와 함께 보세요.")
    return [(h["path"], image_caption(h)) for h in (shown or hits[:IMG_TOP_K])], info


def ui_compare(question, selected):
    if not KB.chunks or not question.strip() or not selected:
        return "문서와 질문을 확인해 주세요.", ""
    hits = {m: retrieve(question, m, selected) for m in MODES}
    answers = generate([build_messages(question, hits[m]) for m in MODES], ANSWER_TOKENS)
    return tuple(f"### {MODES[m]}\n\n{a}\n\n---\n\n{sources_md(hits[m], m)}" for m, a in zip(MODES, answers))


with gr.Blocks(title="DRAG 3주차 RAG 챗봇 + SigLIP", fill_height=True) as demo:
    with gr.Sidebar(label="문서 · 설정", open=True, width=340):
        gr.Markdown("### 📚 문서")
        urls_in = gr.Textbox(label="URL (한 줄에 하나)", lines=3, placeholder="https://ko.wikipedia.org/wiki/...")
        files_in = gr.File(label="PDF 업로드", file_count="multiple", file_types=[".pdf"], height=110)
        add_btn = gr.Button("문서 추가", variant="primary", size="sm")
        add_msg = gr.Markdown()
        doc_sel = gr.CheckboxGroup(choices=doc_choices(), value=[s for _, s in doc_choices()],
                                   label="검색할 문서 (체크한 문서에서만 찾음)")
        del_btn = gr.Button("체크한 문서 삭제", size="sm")
        gr.Markdown("### ⚙️ 설정")
        gr.Markdown("### 🖼️ 이미지 (SigLIP)")
        imgs_in = gr.File(label="이미지 업로드", file_count="multiple", file_types=["image"], height=110)
        add_img_btn = gr.Button("이미지 추가", variant="primary", size="sm")
        img_msg = gr.Markdown(f"🖼️ 전체 {len(IMB.items)}장")
        img_gallery = gr.Gallery(value=[(it["path"], it["name"]) for it in IMB.items], columns=4, height=150,
                                 show_label=False, allow_preview=False)
        clear_img_btn = gr.Button("이미지 모두 지우기", size="sm")
        gr.Markdown("### ⚙️ 설정")
        rerank_on = gr.Checkbox(value=True, label="Reranking 사용 (끄면 2주차 기본 검색)")
        images_on = gr.Checkbox(value=True, label="답변과 함께 관련 이미지 보여주기")
        min_prob = gr.Slider(0.0, 0.5, value=IMG_MIN_PROB, step=0.01, label="이미지 최소 '짝일 확률'")
        gr.Markdown(f"<small>LLM `{MODEL_ID}` · 임베딩 `{EMB_ID}` · 이미지 `{SIGLIP_ID}`</small>")

    with gr.Tab("💬 채팅"):
        chatbot = gr.Chatbot(label="DRAG 챗봇 (문서 RAG + SigLIP 이미지)", height=520, buttons=["copy"],
                             placeholder="문서·이미지를 추가하고 질문해 보세요.")
        with gr.Row():
            msg_in = gr.Textbox(placeholder="질문을 입력하고 Enter", show_label=False, scale=8, autofocus=True)
            send_btn = gr.Button("보내기", variant="primary", scale=1)
        with gr.Row():
            gr.Examples(["RAG 모델에서 retriever와 generator는 각각 무엇을 사용했나?",
                         "대형 언어 모델의 한계는 무엇인가?", "소파 위에서 자는 고양이 두 마리"],
                        inputs=msg_in, label="예시 질문")
            clear_btn = gr.ClearButton([chatbot, msg_in], value="🧹 대화 지우기", size="sm")

    with gr.Tab("🖼️ 이미지 검색"):
        with gr.Row():
            img_q = gr.Textbox(label="찾을 이미지를 문장으로", placeholder="예: 밤에 발사되는 로켓", scale=8)
            img_btn = gr.Button("찾기", variant="primary", scale=1)
        img_info = gr.Markdown()
        img_out = gr.Gallery(columns=3, height=460, label="SigLIP 검색 결과 (확률 높은 순)")

    with gr.Tab("⚖️ 기본 vs Reranking 비교"):
        cmp_in = gr.Textbox(label="질문", placeholder="같은 질문을 두 방식으로 답해 봅니다")
        cmp_btn = gr.Button("비교하기", variant="primary")
        with gr.Row():
            cmp_base = gr.Markdown()
            cmp_rr = gr.Markdown()

    add_btn.click(ui_add_docs, [urls_in, files_in], [add_msg, doc_sel, urls_in, files_in])
    del_btn.click(ui_remove_docs, [doc_sel], [add_msg, doc_sel])
    add_img_btn.click(ui_add_images, [imgs_in], [img_msg, img_gallery, imgs_in])
    clear_img_btn.click(ui_clear_images, None, [img_msg, img_gallery])
    for ev in (msg_in.submit, send_btn.click):
        ev(ui_chat, [msg_in, chatbot, rerank_on, doc_sel, images_on, min_prob], [chatbot, msg_in])
    for ev in (img_q.submit, img_btn.click):
        ev(ui_image_search, [img_q, min_prob], [img_out, img_info])
    for ev in (cmp_in.submit, cmp_btn.click):
        ev(ui_compare, [cmp_in, doc_sel], [cmp_base, cmp_rr])

demo.queue().launch(share=True, debug=False)